### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

### Load data

In [ ]:
df = pd.read_csv("../data/simulated/hourly_energy_data.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")

df.head()


### Create time columns

In [ ]:
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour

df.head()


### Define process

In [ ]:
PROCESS_DURATION_HOURS = 4
ENERGY_USUAGE_PER_HOUR_KWH = 120

STANDARD_START_HOUR = 8
STANDARD_END_HOUR = 12


### Function: find the cheapest 4-hrs window per day

In [ ]:
def find_cheapest_window(day_df, duration_hours=4):
    day_df = day_df.sort_values("timestamp").reset_index(drop=True)

    windows = []

    for i in range(len(day_df) - duration_hours + 1):
        window = day_df.iloc[i:i + duration_hours]

        start_time = window["timestamp"].iloc[0]
        end_time = window["timestamp"].iloc[-1] + pd.Timedelta(hours=1)

        avg_price = window["electricity_price"].mean()
        total_price = window["electricity_price"].sum()

        windows.append({
            "date": start_time.date(),
            "optimized_start": start_time,
            "optimized_end": end_time,
            "optimized_avg_price": avg_price,
            "optimized_price_sum": total_price 
        })

    return pd.DataFrame(windows).sort_values("optimized_price_sum").iloc[0] 

### Find the cheapest window for each day

In [ ]:
optimized_results = (
    df
    .groupby("date")
    .apply(find_cheapest_window, duration_hours=PROCESS_DURATION_HOURS)
    .reset_index(drop=True)
)

optimized_results.head()

### Calculate costs for optimized running

In [ ]:
optimized_results["optimized_cost"] = (
    optimized_results["optimized_price_sum"] * ENERGY_USUAGE_PER_HOUR_KWH
)

optimized_results.head()

### Standard running 08:00-12:00

In [ ]:
standard_df = df[
    (df["hour"] >= STANDARD_START_HOUR) &
    (df["hour"] < STANDARD_END_HOUR) 
]

standard_results = (
    standard_df
    .groupby("date")
    .agg(
        standard_start=("timestamp", "min"),
        standard_end=("timestamp", "max"),
        standard_avg_price=("electricity_price", "mean"),
        standard_price_sum=("electricity_price", "sum"),
    )
    .reset_index()
)

standard_results["standard_end"] = standard_results["standard_end"] + pd.Timedelta(hours=1)

standard_results.head()

### Calculate costs for standard running

In [ ]:
standard_results["standard_cost"] = (
    standard_results["standard_price_sum"] * ENERGY_USUAGE_PER_HOUR_KWH
)

standard_results.head()

### Compare standard against optimization

In [ ]:
comparison = optimized_results.merge(
    standard_results,
    on="date",
    how="inner"
)

comparison["saving"] = comparison["standard_cost"] - comparison["optimized_cost"]

comparison["saving_percent"] = (
    comparison["saving"] / comparison["standard_cost"] * 100
)

comparison.head()

### Conclussion

In [ ]:
total_standard_cost = comparison["standard_cost"].sum()
total_optimized_cost = comparison["optimized_cost"].sum()
total_saving = comparison["saving"].sum()
total_saving_percent = total_saving / total_standard_cost * 100


print("Total cost standard running", round(total_standard_cost, 2))
print("Total cost optimized running", round(total_optimized_cost, 2))
print("Total saving", round(total_saving, 2))
print("Saving in percent", round(total_saving_percent, 2), "%")

### Show the best time window per day

In [ ]:
comparison[
    [
        "date",
        "standard_start",
        "standard_end",
        "standard_cost",
        "optimized_start",
        "optimized_end",
        "optimized_cost",
        "saving",
        "saving_percent"
    ]
].head(16)

### Visualize daily savings

In [ ]:
plt.figure(figsize=(14, 5))

sns.lineplot(
    data=comparison,
    x="date",
    y="saving"
)

plt.title("Daily savings with optimized cost")
plt.xlabel("Date")
plt.ylabel("Savings in SEK")
plt.xticks(rotation=45)
plt.show()
 

### Compare cost: standard vs optimized

In [ ]:
cost_summary = pd.DataFrame({
    "strategy": ["Standard 08:00-12:00", "Optimized cheapest 4h"],
    "total_cost": [total_standard_cost, total_optimized_cost]
})

plt.figure(figsize=(8, 5))

sns.barplot(
    data=cost_summary,
    x="strategy",
    y="total_cost"
)

plt.title("Total cost: standard vs optimized running")
plt.xlabel("Strategy")
plt.ylabel("Total cost")
plt.show()

## Conclussion

- The baseline-model finds the cheapest coherent 4h-window daily. 
- It compares towards a constant standard running between 08:00-12:00.
- The difference between standard cost and optimized cost becomes the estimated saving. 
- This is a simple but effective first version of the optimization logic. 